In [70]:
import pandas as pd

from get_market_cap_by_ticker import get_market_cap_by_ticker
from DATA.stock_invest_function import *
from get_fs_data_by_ticker import extract_quarterly_fs_data
from get_hscode_processed_data import get_hscode_processed_data
from get_revenue_export_joined_table import get_revenue_export_joined_table
from sarima_endog_forecast import forecast_endog_with_optional_exog
from sarima_endog_forecast import forecast_endog_fill_tail
from get_forecasted_revenue_df import build_forecast_df_from_out
from revenue_forecast_all_package import *
from get_revenue_ttm_df import get_revenue_ttm_df
from get_psr_from_mc_and_rev import build_psr_series
from psr_forecast_runner import forecast_psr_all_models

# 1) DB 접속정보
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

ticker = "A035420"
hs_code = None

df_mc = get_market_cap_by_ticker(db_info, ticker)

df_rev = extract_quarterly_fs_data(
    db_info=db_info,
    table_name="korea_fs_data",          # 실제 테이블명으로 교체
    target_indicator="매출액(천원)",       # 원하는 지표명
    ticker= ticker,                     # 원하는 종목코드
)

# df_exog['monthly_raw']   # 월별 데이터
# df_exog['quarterly']     # 분기 데이터
# df_exog['exog']

df_export = get_hscode_processed_data(db_info, hs_code = hs_code)
df_exog = df_export['quarterly']

combined_df, final_combined_data, forecast_df = get_revenue_export_joined_table(
    df_rev=df_rev,
    df_exog=df_exog,
    join_how="outer",
    fill_exog="ffill"   # 필요시
)

# combined_df, final_combined_data, forecast_df = get_revenue_export_joined_table(
#     df_rev=df_rev,
#     df_exog=df_exog,
#     join_how="right",
#     fill_exog="ffill"   # 필요시
# )

# combined_df: Date 인덱스 + ['endog_var','exog_var'] (exog_var는 없어도 됨)
# 예: horizon=4 (분기 4개 또는 월 4개)
out = forecast_endog_with_optional_exog(
    combined_df=final_combined_data,
    horizon=5,          # ⬅️ 예측 기간 설정
    hs_code= None,   # ⬅️ None이면 exog_var 사용 안함
    # seasonal_period=4 # 직접 지정도 가능(미지정시 자동 추론)
)
rev_forecast_with_noexog = out['forecast']

out2 = forecast_endog_fill_tail(final_combined_data, hs_code= hs_code)
rev_forecast_with_exog = out2['forecast']

rev_sarima_noexog = build_forecast_df_from_out(out, combined_df=final_combined_data)
rev_sarima_exog = build_forecast_df_from_out(out2, combined_df=final_combined_data)

rev_ets_df     = forecast_revenue_ets(final_combined_data, horizon=5)
rev_prophet_df = forecast_revenue_prophet(final_combined_data, horizon=5)   # Prophet 미설치면 에러
rev_lstm_df    = forecast_revenue_lstm(final_combined_data, horizon=5, lookback=12)
rev_theta_df   = forecast_revenue_theta(final_combined_data, horizon=5)

rev_final = get_revenue_ttm_df(
    df_rev=df_rev,
    rev_sarima_noexog=rev_sarima_noexog,
    rev_sarima_exog=rev_sarima_exog,
    rev_ets_df=rev_ets_df,
    rev_prophet_df=rev_prophet_df,
    rev_theta_df=rev_theta_df,
    rev_lstm_df=rev_lstm_df  # 또는 ref_lstm_df
    # forecast_col_map={"prophet": "yhat"}  # 필요시 예측 컬럼 강제 지정
)

psr_df = build_psr_series(df_mc=df_mc, df_rev=df_rev)

# -------------------------------------------
# psr 예측 파이프라인 (Py3.9 호환)
# -------------------------------------------

# -------- 사용 예시 --------
psr_df = psr_df  # index=DatetimeIndex, column='psr'
exog_df = final_combined_data[['exog_var']]
horizon = 13  # 원하는 예측기간
fc_table = forecast_psr_all_models(psr_df, horizon=horizon, exog_df=exog_df)

from valuation_forecast import compute_valuation_forecast

# value_start_date 를 None 으로 두면 “현재 달 + 1개월”이 자동 적용됩니다. (예: 오늘이 2025-10이면 2025-11)
valuation_forecast_result = compute_valuation_forecast(
    fc_table=fc_table,
    rev_final=rev_final,
    value_start_date=None  # 또는 '2025-11' 같이 직접 지정
)

print(valuation_forecast_result.tail())

✅ A035420 시가총액 4,204건 조회 완료
[메모리] forecast_sarima 실행 전: 723.10 MB
[메모리] find_best_sarima_params 실행 전: 723.10 MB
[메모리] find_best_sarima_params 실행 후: 722.45 MB (변화: -0.65 MB)
[메모리] forecast_sarima 실행 후: 722.45 MB (변화: -0.65 MB)
[메모리] find_best_sarima_params 실행 전: 722.45 MB
[메모리] find_best_sarima_params 실행 후: 722.46 MB (변화: +0.02 MB)


KeyError: "None of [DatetimeIndex(['2025-09-30'], dtype='datetime64[ns]', freq=None)] are in the [index]"

In [78]:
rev_forecast_with_exog

array([8.00717445e+10, 7.47790172e+10, 7.74349304e+10, 7.57077955e+10,
       7.92084810e+10])

In [62]:
from upload_valuation_longform import upload_valuation_longform

upload_valuation_longform(
    valuation_forecast_result=valuation_forecast_result,
    fc_table=fc_table,
    rev_final=rev_final,
    psr_df=psr_df,
    ticker=ticker,
    db_info=db_info,
    table_name="Korea_company_valuation_ver2",
    forecast_date=None   # None이면 오늘 날짜로 입력
)

[OK] 1330 rows upserted into Korea_company_valuation_ver2 for ticker=A005930 (forecast_date=2025-10-29).


In [44]:
# combined_df, final_combined_data, forecast_df = get_revenue_export_joined_table(
#     df_rev=df_rev,
#     df_exog=df_exog,
#     join_how="right",
#     fill_exog="ffill"   # 필요시
# )


In [45]:

# combined_df: Date 인덱스 + ['endog_var','exog_var'] (exog_var는 없어도 됨)
# 예: horizon=4 (분기 4개 또는 월 4개)
# out = forecast_endog_with_optional_exog(
#     combined_df=final_combined_data,
#     horizon=5,          # ⬅️ 예측 기간 설정
#     hs_code= None,   # ⬅️ None이면 exog_var 사용 안함
#     # seasonal_period=4 # 직접 지정도 가능(미지정시 자동 추론)
# )
# rev_forecast_with_noexog = out['forecast']

[메모리] forecast_sarima 실행 전: 609.48 MB
[메모리] find_best_sarima_params 실행 전: 609.48 MB
[메모리] find_best_sarima_params 실행 후: 611.74 MB (변화: +2.26 MB)
[메모리] forecast_sarima 실행 후: 611.74 MB (변화: +2.26 MB)


In [46]:
# out2 = forecast_endog_fill_tail(final_combined_data, hs_code="854232")
# rev_forecast_with_exog = out2['forecast']

[메모리] find_best_sarima_params 실행 전: 611.74 MB
[메모리] find_best_sarima_params 실행 후: 612.12 MB (변화: +0.38 MB)


In [47]:
# rev_sarima_noexog = build_forecast_df_from_out(out, combined_df=final_combined_data)
# rev_sarima_exog = build_forecast_df_from_out(out2, combined_df=final_combined_data)

In [48]:
# final_combined_data: get_revenue_export_joined_table(...)에서 받은 것
# rev_ets_df     = forecast_revenue_ets(final_combined_data, horizon=5)
# rev_prophet_df = forecast_revenue_prophet(final_combined_data, horizon=5)   # Prophet 미설치면 에러
# rev_lstm_df    = forecast_revenue_lstm(final_combined_data, horizon=5, lookback=12)
# rev_theta_df   = forecast_revenue_theta(final_combined_data, horizon=5)

[메모리] forecast_ets 실행 전: 612.12 MB


10:24:41 - cmdstanpy - INFO - Chain [1] start processing


[메모리] forecast_ets 실행 후: 612.15 MB (변화: +0.03 MB)
[메모리] forecast_prophet 실행 전: 612.15 MB


10:24:41 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_prophet 실행 후: 612.50 MB (변화: +0.35 MB)
[메모리] forecast_lstm 실행 전: 612.50 MB
[메모리] forecast_lstm 실행 후: 638.39 MB (변화: +25.89 MB)
[메모리] forecast_theta 실행 전: 638.39 MB
[메모리] forecast_theta 실행 후: 638.39 MB (변화: +0.00 MB)


In [49]:
# from get_revenue_ttm_df import get_revenue_ttm_df
#
# rev_final = get_revenue_ttm_df(
#     df_rev=df_rev,
#     rev_sarima_noexog=rev_sarima_noexog,
#     rev_sarima_exog=rev_sarima_exog,
#     rev_ets_df=rev_ets_df,
#     rev_prophet_df=rev_prophet_df,
#     rev_theta_df=rev_theta_df,
#     rev_lstm_df=rev_lstm_df  # 또는 ref_lstm_df
#     # forecast_col_map={"prophet": "yhat"}  # 필요시 예측 컬럼 강제 지정
# )

### PSR 측정

In [50]:
# from get_psr_from_mc_and_rev import build_psr_series
# from psr_forecast_runner import forecast_psr_all_models
#
# psr_df = build_psr_series(df_mc=df_mc, df_rev=df_rev)

In [51]:
# -------------------------------------------
# psr 예측 파이프라인 (Py3.9 호환)
# -------------------------------------------

# -------- 사용 예시 --------
# psr_df = psr_df  # index=DatetimeIndex, column='psr'
# exog_df = final_combined_data[['exog_var']]
# horizon = 13  # 원하는 예측기간
# fc_table = forecast_psr_all_models(psr_df, horizon=horizon, exog_df=exog_df)


[메모리] forecast_sarima 실행 전: 638.39 MB
[메모리] find_best_sarima_params 실행 전: 638.39 MB
[메모리] find_best_sarima_params 실행 후: 639.62 MB (변화: +1.23 MB)
[메모리] forecast_sarima 실행 후: 639.62 MB (변화: +1.24 MB)
[메모리] forecast_ets 실행 전: 639.62 MB


10:25:00 - cmdstanpy - INFO - Chain [1] start processing
10:25:00 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_ets 실행 후: 639.67 MB (변화: +0.05 MB)
[메모리] forecast_prophet 실행 전: 639.67 MB
[메모리] forecast_prophet 실행 후: 641.28 MB (변화: +1.61 MB)
[메모리] forecast_lstm 실행 전: 641.28 MB
[메모리] forecast_lstm 실행 후: 664.37 MB (변화: +23.09 MB)
[메모리] forecast_theta 실행 전: 664.37 MB
[메모리] forecast_theta 실행 후: 664.37 MB (변화: +0.00 MB)
[경고] SARIMA exog 예측 실패: exog contains inf or nans


In [52]:
# from valuation_forecast import compute_valuation_forecast
#
# # value_start_date 를 None 으로 두면 “현재 달 + 1개월”이 자동 적용됩니다. (예: 오늘이 2025-10이면 2025-11)
# valuation_forecast_result = compute_valuation_forecast(
#     fc_table=fc_table,
#     rev_final=rev_final,
#     value_start_date=None  # 또는 '2025-11' 같이 직접 지정
# )
#
# print(valuation_forecast_result.tail())


         date  mc_sarima_noexog  mc_sarima_exog        mc_ets    mc_prophet  \
95 2026-07-31      4.879462e+11             NaN  6.805312e+11  2.192711e+11   
96 2026-08-31      4.884669e+11             NaN  6.627236e+11  2.047174e+11   
97 2026-09-30      4.620216e+11             NaN  7.252294e+11  2.173876e+11   
98 2026-10-31      4.589469e+11             NaN  7.453668e+11  2.362648e+11   
99 2026-11-30      4.589760e+11             NaN  7.424774e+11  2.081510e+11   

         mc_lstm      mc_theta  
95  3.898856e+11  5.613059e+11  
96  3.696637e+11  5.619110e+11  
97  4.188820e+11  5.652055e+11  
98  3.934901e+11  5.658135e+11  
99  3.760337e+11  5.664215e+11  


### DB에 결과 업로드

In [60]:
from upload_valuation_longform import upload_valuation_longform

upload_valuation_longform(
    valuation_forecast_result=valuation_forecast_result,
    fc_table=fc_table,
    rev_final=rev_final,
    psr_df=psr_df,
    ticker=ticker,
    db_info=db_info,
    table_name="Korea_company_valuation_ver2",
    forecast_date=None   # None이면 오늘 날짜로 입력
)

[OK] 1330 rows upserted into Korea_company_valuation_ver2 for ticker=A000660 (forecast_date=2025-10-29).


In [59]:
psr_df.tail(5)

,psr
date,
2025-06-30,3.866495
2025-07-31,3.621530
2025-08-31,3.283688
2025-09-30,4.241934
2025-10-31,6.359841
